In [ ]:
import torch
import numpy as np
import meent
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

In [ ]:
start = time.time()

In [ ]:
P = 1000
t = 500
h = 500
N_L = 1
N_C = 64
pixel_min_width = P/N_C
pixel_min_height = t/N_L

res_x = N_C
res_z = 15

dy = t/N_L/res_z

wl_list = list(range(650, 751, 10))

In [ ]:
n_air = 1.0

backend = 2  # torch
device = 0
pol = 1 # 0: TE, 1: TM


theta = np.array([0]) * torch.pi / 180  # angle of incidence
phi = 0 * torch.pi / 180  # angle of rotation
period = torch.tensor([P])  # length of the unit cell. Here it's 1D.

fourier_order = [81] # Chosen based on FO sweep for resonant structure
type_complex = torch.complex128

n_mat = 2.0

k_list = [0, 0.01]

thickness = torch.tensor([dy * res_z for i in range (1)] + [pixel_min_height for layer in range(N_L)] + [dy * res_z for i in range(5)]) # thickness of each layer, from top to bottom.

In [ ]:
def return_device(patterning):
    Layer_air =  n_air * torch.ones(1,1,N_C)
    Layer_patterning = torch.from_numpy(np.expand_dims(patterning, axis=1))
    Layer_detector =  n_air * torch.ones(5,1,N_C)

    ucell = torch.cat((Layer_air,Layer_patterning,Layer_detector))
    return ucell


In [ ]:
result_folder_path = r'D:\Datasets\DIRTL_Manuscript'
os.makedirs(result_folder_path, exist_ok=True)
input_patterning_folder = os.path.join(result_folder_path, 'patternings_multiwl_test')
output_result_folder = os.path.join(result_folder_path, 'true_results_multiwl_test')
os.makedirs(input_patterning_folder, exist_ok=True)
os.makedirs(output_result_folder, exist_ok=True)

In [ ]:
num_samples = 1000

for epoch in tqdm(range(num_samples), dynamic_ncols=True, desc="Simulation Complete: ") :

    pattern = np.random.randint(2, size=(N_L, N_C))

    for tmp in range(len(k_list)):

        k = k_list[tmp]

        n_mat = 2.0 - 1j * k
    
        random_array = np.ones((N_L, N_C)) *(n_air) + (n_mat-n_air)* pattern
        ucells = []

        name = str(epoch).zfill(8)
        patterning_path = os.path.join(input_patterning_folder, 'patterning_'+name+'.npy')
        np.save(patterning_path, np.real(random_array))

        field_cell_meent_color = []
        field_cell_meent_Hy_color = []

        for wavelength in wl_list:
            
            wav_len = wavelength

            ucell = return_device(random_array)
            ucells.append(ucell.tolist())
            mee = meent.call_mee(backend=backend, pol=pol, n_top=n_air, n_bot=n_air, theta=0.0, phi=phi, fto=fourier_order, wavelength=wav_len, period=period, thickness=thickness, type_complex=type_complex, device=device, fourier_type=0) # Meent version 0.10.0

            mee.ucell = ucell

            de_ri, de_ti = mee.conv_solve()

            field_cell_meent = mee.calculate_field(res_x=res_x,res_z=res_z)
            observation = res_z * (1+ N_L + 5)
            field_cell_meent_Hy = field_cell_meent[:observation,0,:,0]
            field_cell_meent_Hy_color.append(field_cell_meent_Hy.numpy())

        field_cell_meent_Hy_color = np.array(field_cell_meent_Hy_color)
        
        result_path = os.path.join(output_result_folder, 'result_Hy_'+name+'k'+f"{k:.4f}" +'.npy')
        np.save(result_path, field_cell_meent_Hy_color)


end_time = time.time()

elapsed_time = end_time - start
print(elapsed_time)

Simulation Complete: 100%|██████████| 1000/1000 [1:51:58<00:00,  6.72s/it]

6718.515138864517
